# SIH26006 — B2.5: Economic / Directional Validation Gate

> **B3 is FROZEN. `decision_engine.py` is NOT touched until B2.5 gives a clear verdict.**

## What this notebook does

Takes the B2 winning pairs and runs a final gate before production:

1. **Strongest-naive re-comparison** on the same test rows
   - Δ-Ridge vs Persistence, PrevDirection, 7dTrend
   - Macro-F1 and per-class F1 at ±2% threshold

2. **Direction from Δ prediction** — `pred_pct = delta_pred / y_base`
   - UP if > +2%, DOWN if < −2%, NEUTRAL otherwise
   - F1 vs true ±2% direction label

3. **Economic backtest** (uncertainty-gated)
   - Signal fires only when `|delta_pred| > half_width of 80%-interval`
   - BUY NOW precision, WAIT precision, expected $/day saving
   - Total freight exposure reduction vs always-FLEXIBLE strategy

## B2 results (pre-confirmed)

| Asset | H | Δ R² | Δ sMAPE | Verdict |
|---|---|---|---|---|
| KDCI | 1d | 0.994 | 1.65% | STRONG_WIN |
| KDCI | 7d | 0.862 | 7.94% | WIN |
| Cape | 1d | 0.986 | 3.13% | STRONG_WIN |
| Cape | 7d | 0.746 | 13.76% | WIN |
| Panamax | 1d | 0.995 | 1.27% | STRONG_WIN |
| Supramax | 1d | — | — | WIN |
| Supramax | 7d | 0.911 | — | WIN |
| Supramax | 14d | 0.742 | — | WIN |
| Supramax | 30d | 0.437 | — | WIN |
| Handy | 1d | 0.998 | 0.78% | STRONG_WIN |


In [ ]:
# ================================================================================
# CELL 1: SETUP (pull latest, check B2 outputs exist)
# ================================================================================
import os
import numpy as np
import pandas as pd
np.random.seed(42)

# Navigate to project root
if not os.path.exists('outputs/modeling_dataset.csv'):
    if os.path.exists('FICOS-Platform/outputs/modeling_dataset.csv'):
        %cd FICOS-Platform
    else:
        !git clone https://github.com/SSOHEB/FICOS-Platform.git
        %cd FICOS-Platform

!git pull origin main

print(f'CWD: {os.getcwd()}')
b2_pred   = os.path.exists('outputs/delta_forecast/b2_test_predictions.csv')
b2_scores = os.path.exists('outputs/delta_forecast/b2_comparison_scoreboard.csv')
print(f'B2 predictions : {b2_pred}')
print(f'B2 scoreboard  : {b2_scores}')

if not b2_pred or not b2_scores:
    print('\nWARNING: B2 outputs not found.')
    print('Run the phase_b_delta_forecast notebook (Cells 2+3) first, then come back.')
else:
    print('\nB2 outputs confirmed. Ready to run B2.5.')

In [ ]:
# ================================================================================
# CELL 2: RUN B2.5
# ================================================================================
# Estimated runtime: 1-3 minutes (no model training — uses B2 predictions)
# Outputs:
#   outputs/b25_validation/b25_naive_comparison.csv
#   outputs/b25_validation/b25_economic_backtest.csv
#   outputs/b25_validation/b25_signal_log.csv
!python scratch/b25_economic_validation.py

In [ ]:
# ================================================================================
# CELL 3: DISPLAY — NAIVE COMPARISON TABLE
# ================================================================================
import pandas as pd

if os.path.exists('outputs/b25_validation/b25_naive_comparison.csv'):
    df_naive = pd.read_csv('outputs/b25_validation/b25_naive_comparison.csv')

    print('=' * 70)
    print('B2.5.1 — DIRECTION MACRO-F1 @ ±2%: Δ-Ridge vs ALL Naive Baselines')
    print('=' * 70)

    # Pivot: asset-horizon rows, method columns
    pivot = df_naive.pivot_table(
        index=['asset','horizon'], columns='method',
        values='macro_F1', aggfunc='first'
    ).reset_index()
    pivot.columns.name = None

    method_cols = [c for c in ['DeltaRidge','Persistence','PrevDirection','7dTrend'] if c in pivot.columns]
    pivot['Best_naive'] = pivot[method_cols[1:]].max(axis=1)
    pivot['Δ_margin']  = pivot['DeltaRidge'] - pivot['Best_naive']
    pivot['Δ_wins']    = pivot['Δ_margin'] > 0

    display(pivot[['asset','horizon'] + method_cols + ['Best_naive','Δ_margin','Δ_wins']])

    n_beats = pivot['Δ_wins'].sum()
    print(f'\nΔ-Ridge beats ALL naive: {n_beats}/{len(pivot)} pairs')
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 4: DISPLAY — ECONOMIC BACKTEST SUMMARY
# ================================================================================
if os.path.exists('outputs/b25_validation/b25_economic_backtest.csv'):
    df_econ = pd.read_csv('outputs/b25_validation/b25_economic_backtest.csv')

    print('=' * 70)
    print('B2.5.2 — ECONOMIC BACKTEST: Uncertainty-Gated Signal Performance')
    print('=' * 70)

    # Core economic columns
    eco_cols = [
        'asset','horizon','win_verdict_b2',
        'n_buy_signals','buy_precision%','buy_expected_pnl_$/day',
        'n_wait_signals','wait_precision%','wait_expected_pnl_$/day',
        'avg_pnl_per_test_day','total_pnl_$/day_sum',
        '80pct_interval_width','high_conf_signals%'
    ]
    display(df_econ[[c for c in eco_cols if c in df_econ.columns]])

    print('\n--- INTERPRETATION ---')
    print('avg_pnl_per_test_day : avg freight rate exposure saved vs FLEXIBLE ($/day) per test date')
    print('buy_precision%       : % of BUY_NOW signals where rate actually went up (signal was correct)')
    print('wait_precision%      : % of WAIT signals where rate actually went down')
    print('high_conf_signals%   : % of test dates where |pred_delta| > uncertainty half-width')
    print('80pct_interval_width : P10-P90 interval from validation residuals — respects Cape/Panamax uncertainty')
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 5: DISPLAY — B3 PROMOTION DECISION
# ================================================================================
if os.path.exists('outputs/b25_validation/b25_economic_backtest.csv'):
    df_econ = pd.read_csv('outputs/b25_validation/b25_economic_backtest.csv')
    df_naive = pd.read_csv('outputs/b25_validation/b25_naive_comparison.csv')

    print('=' * 70)
    print('B2.5 GATE VERDICT — B3 PROMOTION DECISION')
    print('=' * 70)

    pivot = df_naive.pivot_table(
        index=['asset','horizon'], columns='method',
        values='macro_F1', aggfunc='first'
    ).reset_index()
    pivot.columns.name = None
    method_cols = [c for c in ['DeltaRidge','Persistence','PrevDirection','7dTrend'] if c in pivot.columns]
    pivot['beats_all_naive'] = pivot['DeltaRidge'] > pivot[method_cols[1:]].max(axis=1)

    summary = df_econ.merge(pivot[['asset','horizon','beats_all_naive']], on=['asset','horizon'])

    promoted = summary[summary['beats_all_naive']]
    not_prom = summary[~summary['beats_all_naive']]

    print(f'\n  PROMOTED TO B3 ({len(promoted)}/{len(summary)} pairs):')
    for _, r in promoted.iterrows():
        bp = f"{r['buy_precision%']:.0f}%" if pd.notna(r['buy_precision%']) else 'N/A'
        wp = f"{r['wait_precision%']:.0f}%" if pd.notna(r['wait_precision%']) else 'N/A'
        print(f'    ★ {r["asset"].upper():>9} {r["horizon"]:>4}  '
              f'BUY_prec={bp}  WAIT_prec={wp}  '
              f'avg_PnL/day={r["avg_pnl_per_test_day"]:+.1f}')

    if len(not_prom) > 0:
        print(f'\n  NOT promoted (naive still stronger):')
        for _, r in not_prom.iterrows():
            print(f'    ✗ {r["asset"].upper():>9} {r["horizon"]:>4}')

    total_pnl = promoted['total_pnl_$/day_sum'].sum()
    print(f'\n  Combined freight exposure reduction across promoted pairs: {total_pnl:+,.0f} $/day-sum over test period')

    if len(promoted) >= len(summary) * 0.5:
        print('\n  ✓ PROCEED TO B3: majority of winning pairs survive final gate.')
        print('    Asset×horizon-selective integration into decision engine.')
    else:
        print('\n  ✗ MIXED: Review individual pairs before B3. Consider feature engineering.')
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 6: SIGNAL LOG SAMPLE (audit)
# ================================================================================
if os.path.exists('outputs/b25_validation/b25_signal_log.csv'):
    df_sig = pd.read_csv('outputs/b25_validation/b25_signal_log.csv')

    print('=' * 70)
    print('B2.5 SIGNAL LOG — Sample BUY_NOW and WAIT signals')
    print('=' * 70)

    for asset_h in df_sig[['asset','horizon']].drop_duplicates().values[:3]:
        asset, horizon = asset_h
        sub = df_sig[(df_sig['asset'] == asset) & (df_sig['horizon'] == horizon)]
        fired = sub[sub['signal'] != 'FLEXIBLE']
        if len(fired) == 0:
            continue
        print(f'\n{asset.upper()} {horizon} — Fired signals (first 10):')
        display(fired[['date','y_base','y_true','delta_pred','pred_pct','act_pct',
                        'signal','sig_unc_ratio','economic_result_$/day']].head(10))
else:
    print('Run Cell 2 first.')